# Virtual Arena

This notebook is an implementation of a complete workflow in a virtual arena using CoppeliaSim.

---

In [1]:
# Importing modules...
import numpy as np
import scipy as sp

import sys
sys.path.append('../..') # Go back to base directory

from modules.plot.viewer3d import Viewer3D

from modules.integration.client import Client
from modules.integration.coppeliasim.server import CoppeliaSim_Server
from modules.integration.coppeliasim.camera import CoppeliaSim_Camera

# Create server
server = CoppeliaSim_Server(
    server_address=('127.0.0.1', 8888),
    controller_address=('127.0.0.1', 7777)
)

In [2]:
# Data structure for marker triangulation
class Triangulator:
    def __init__(self, multiple_view):

        # Initializing parameters
        self.multiple_view = multiple_view
        self.blobs_lists = [] # Stores every blob sent
        self.blobs_queues = [] # Stores blobs in queues for triangulation
        self.tri_idx = -1

        # Setup configuration
        self.reset()

    def reset(self):
        self.blobs_lists = [[] for _ in range(self.multiple_view.n_cameras)]
        self.blobs_queues = [[] for _ in range(self.multiple_view.n_cameras)]

    def save(self, id, frame_idx, blobs):
        # Log data
        self.blobs_lists[id].append(
            (blobs, frame_idx)
        )  # Add blobs to list

    def full_vision(self):
        frame_idxs = [list(zip(*blobs_list))[1] for blobs_list in self.blobs_lists]
        sync_frame_idxs = list(set.intersection(*map(set, frame_idxs)))
        sync_frame_idxs.sort()

        sync_blobs = []
        for blobs_list in self.blobs_lists:
            sync_blobs.append([frame_data[0] for frame_data in blobs_list if frame_data[1] in sync_frame_idxs])

        return sync_blobs

    def triangulate(self, reference, frame_idx, blobs_reference):
        # Log data
        self.save(reference, frame_idx, blobs_reference)

        # Do not triangulate if triangulation is ahead from received data
        if frame_idx <= self.tri_idx:
            return None

        available_data = []
        for queue_id, blob_queue in enumerate(self.blobs_queues):
            # Ignore list on received ID
            if queue_id == reference:
                continue

            # Do not search blob queue is empty
            if not len(blob_queue):
                continue

            try:
                queue_position = list(zip(*blob_queue))[1].index(
                    frame_idx
                )  # Get queue position of the frame index

            except:  # Did not find frame index, go to next queue
                continue

            available_data.append(
                (
                    blob_queue[queue_position][0],  # Get blobs
                    queue_id,  # Get ID of correspondent queue
                )
            )

        # If didn't find any possible triangulation
        if not available_data:
            self.blobs_queues[reference].append(
                (blobs_reference, frame_idx)
            )  # Add blobs to queue
            return None

        # If there is data available
        for triangulation_candidate in available_data:
            # Try to triangulate received data to candidate
            blobs_auxiliary, auxiliary = triangulation_candidate

            blobs_pair = [
                blobs_reference,
                blobs_auxiliary,
            ]

            triangulated_markers = self.multiple_view.triangulate_by_pair(
                (reference, auxiliary), blobs_pair
            )

            # Triangulation is not reliable
            if np.isnan(triangulated_markers).any():
                continue # Try next triangulation candidate

            # If triangulation was possible, clear queue
            for queue_id, blob_queue in enumerate(self.blobs_queues):
                self.blobs_queues[queue_id] = [
                    queue_element
                    for queue_element in blob_queue
                    if queue_element[1] > frame_idx
                ]  # Only points after triangulation remains

            # Update triangulation index
            self.tri_index = frame_idx

            # Return successfully triangulated markers
            return triangulated_markers
        
        return None # No triangulation was possible with available data

# Loading Calibration Data

All `Camera` and `Client` objects will be loaded from a previous calibration in disk - jump to the "Standard Capture" cell and run it and the cells below to start a capture routine. 

If you wish to perform a calibration, run the "Setting the Scene" cells and below.

---

In [3]:
"""server.load_calibration(r"calibration/24-12-14/14-56-52")

# Request scene with the associated server clients
if not server.request_scene():
    sys.exit() # Scene request failed!"""

'server.load_calibration(r"calibration/24-12-14/14-56-52")\n\n# Request scene with the associated server clients\nif not server.request_scene():\n    sys.exit() # Scene request failed!'

# Setting the Scene

All `Camera` and `Client` objects will be instanciated right along with the `Server` object. A CoppeliaSim scene will be requested given the arbitrary camera parameters.

---

In [4]:
n_clients = 4 # Number of clients in the arena
clients = []  # Clients list

# Object matrix of Camera 0
base_matrix = np.array([[-7.07106781e-01,  5.00000000e-01, -5.00000000e-01, 2.50000000e+00],
                        [ 7.07106781e-01,  5.00000000e-01, -5.00000000e-01, 2.50000000e+00],
                        [ 1.46327395e-13, -7.07106781e-01, -7.07106781e-01, 2.50000000e+00]])

# Create clients
for ID in range(n_clients):
    # Spread all cameras uniformely in a circle around the arena
    R = np.array(sp.spatial.transform.Rotation.from_euler('z', (360 / n_clients) * ID, degrees=True).as_matrix())
    pose = np.vstack((R @ base_matrix,
                      np.array([0, 0, 0, 1])))

    # Generate associated camera model
    camera = (CoppeliaSim_Camera(resolution=(1080, 1080), 
                                 fov_degrees=60.0,     
                                 pose=pose,
                                 distortion_model='fisheye',
                                 distortion_coefficients=np.array([0.395, 0.633, -2.417, 2.110]),
                                 snr_dB=13
                                 ))
    
    clients.append(Client(camera=camera))

server.update_clients(clients=clients)

# Request scene with the associated server clients
if not server.request_scene():
    sys.exit() # Scene request failed!

[SERVER] Wrapping up CoppeliaSim scene info
[SERVER] Scene info sent
[SERVER] Scene set!


# Extrinsic Calibration

The Cameras' Extrinsic Parameters will be estimated. For that, a calibration routine will be requested and the data will be post-processed for the parameter estimation.

---

In [5]:
# Capture specifications
blob_count = 3 # Number of expected markers
capture_time = 30.0 # In seconds
triangulator = Triangulator(server.multiple_view)

# Request capture (start simulation)
if not server.request_sync_calibration(capture_time):
    sys.exit() # Capture request failed!

# Wait for client identification
server.register_clients()

[SERVER] Extrinsic Calibration info sent
[SERVER] Extrinsic Calibration confirmed!
[SERVER] Waiting for clients...
	Client 0 registered
	Client 1 registered
	Client 2 registered
	Client 3 registered
[SERVER] All clients registered!


In [6]:
verbose = False

timeout = 5 # In seconds
server.udp_socket.settimeout(timeout) # Set server timeout
print(f'[SERVER] Timeout set to {timeout} seconds\n')

# Receiving messages
while True: 
    # Wait for message - Event guided!
    try:
        message_bytes, address = server.udp_socket.recvfrom(server.buffer_size)

    except TimeoutError:
        print('\n[SERVER] Timed Out!')
        break # Close capture loop due to timeout

    except ConnectionResetError:
        print('\n[SERVER] Connection Reset!')
        continue # Jump to wait for the next message
    
    # Check if client exists
    try:
        ID = server.client_addresses[address] # Client Identifier
    
    except:
        if verbose: print('> Client not recognized')

        continue # Jump to wait for the next message
    
    # Show sender
    if verbose: print(f'> Received message from Client {ID} ({address[0]}, {address[1]})')

    # Save message
    server.clients[ID].message_log.append(message_bytes)

# Post-processing
for ID, client in enumerate(server.clients):
    # Parse through client's message history
    for message_bytes in client.message_log: 
        # Decode message
        try:
            message = np.frombuffer(message_bytes, dtype=np.float32)

        except:
            if verbose: print('> Couldn\'t decode message')

            continue # Jump to the next message

        # Empty message
        if not message.size:
            if verbose: print('\tEmpty message')

            continue # Jump to the next message

        # Extracting the message's frame index
        frame_idx = int(message[-1])

        # Valid message is [u, v, A] per blob, PTS and frame index
        if message.size !=  3 * blob_count + 2:

            if message.size == 2: # Only PTS
                if verbose: print(f'\tNo blobs were detected - {frame_idx} s')

            else: 
                if verbose: 
                    print(f'\tWrong blob count or corrupted message')
                    print(f'\tCorrupted Message: {message}')

            continue # Jump to the next message

        # Extracting blob data (coordinates & area)
        blob_data = message[:-2].reshape(-1, 3) # All but last two elements

        # Extracting centroids
        blob_centroids = blob_data[:,:2] # Ignoring their area

        # Undistorting blobs centroids
        undistorted_blobs = client.camera.undistort_points(blob_centroids)          

        # Print blobs
        if verbose:
            print(f'\tDetected Blobs - {frame_idx} s')
            print('\t' + str(blob_data).replace('\n', '\n\t'))

        # Save data
        triangulator.save(ID, frame_idx, undistorted_blobs)

[SERVER] Timeout set to 5 seconds


[SERVER] Timed Out!


In [7]:
# Calibrate multiple view
wand_distances = np.array([5e-2, 10e-2, 15e-2]) # In meters

wand_blobs = triangulator.full_vision()

if not server.multiple_view.calibrate(wand_blobs, wand_distances):
    sys.exit() # Calibration failed!

In [8]:
#  Create the Scene Viewer
scene = Viewer3D(title='Calibrated Camera Poses', 
                 size=10)

# Add camera frames to the scene
for ID, camera in enumerate(server.multiple_view.camera_models): 
    scene.add_frame(camera.pose, f'Camera {ID}', axis_size=0.4)

# Plot scene
scene.figure.show(renderer='notebook_connected')

In [9]:
# Perform bundle adjustment
n_observations = 72 # Choose a number multiple of the total number of unique pairs: n_cameras * (n_cameras - 1) / 2
server.multiple_view.bundle_adjustment(wand_blobs, wand_distances, n_observations)

In [10]:
# Create the Scene Viewer
scene = Viewer3D(title='Adjusted Camera Poses', 
                 size=10)

# Add camera frames to the scene
for ID, camera in enumerate(server.multiple_view.camera_models): 
    scene.add_frame(camera.pose, f'Camera {ID}', axis_size=0.4)

# Plot scene
scene.figure.show(renderer='notebook_connected')

# Reference Update

Right after the Extrinsic Calibration, a new reference will be set to be the new scene's canonical frame. For that, a new reference will be requested.

---

In [11]:
# Capture specifications
blob_count = 3 # Number of expected markers
capture_time = 1.0 # In seconds
triangulator = Triangulator(server.multiple_view)

# Request capture (start simulation)
if not server.request_sync_reference(capture_time):
    sys.exit() # Capture request failed!

# Wait for client identification
server.register_clients()

[SERVER] Reference Update info sent
[SERVER] Reference Update confirmed!
[SERVER] Waiting for clients...
	Client 0 registered
	Client 1 registered
	Client 2 registered
	Client 3 registered
[SERVER] All clients registered!


In [12]:
verbose = False

timeout = 5 # In seconds
server.udp_socket.settimeout(timeout) # Set server timeout
print(f'[SERVER] Timeout set to {timeout} seconds\n')

# Breaks in the timeout
while True: 
    # Wait for message - Event guided!
    try:
        message_bytes, address = server.udp_socket.recvfrom(server.buffer_size)

    except TimeoutError:
        print('\n[SERVER] Timed Out!')
        break # Close capture loop due to timeout

    except ConnectionResetError:
        print('\n[SERVER] Connection Reset!')
        continue # Jump to wait for the next message
    
    # Check if client exists
    try:
        ID = server.client_addresses[address] # Client Identifier
    
    except:
        if verbose: print('> Client not recognized')

        continue # Jump to wait for the next message
    
    # Show sender
    if verbose: print(f'> Received message from Client {ID} ({address[0]}, {address[1]})')

    # Save message
    server.clients[ID].message_log.append(message_bytes)

for ID, client in enumerate(server.clients):
    # Parse through client's message history
    for message_bytes in client.message_log: 
        # Decode message
        try:
            message = np.frombuffer(message_bytes, dtype=np.float32)

        except:
            if verbose: print('> Couldn\'t decode message')

            continue # Jump to the next message

        # Empty message
        if not message.size:
            if verbose: print('\tEmpty message')

            continue # Jump to the next message

        # Extracting the message's frame index
        frame_idx = int(message[-1])

        # Valid message is [u, v, A] per blob, PTS and frame index
        if message.size !=  3 * blob_count + 2:

            if message.size == 2: # Only PTS
                if verbose: print(f'\tNo blobs were detected - {frame_idx} s')

            else: 
                if verbose: 
                    print(f'\tWrong blob count or corrupted message')
                    print(f'\tCorrupted Message: {message}')

            continue # Jump to the next message

        # Extracting blob data (coordinates & area)
        blob_data = message[:-2].reshape(-1, 3) # All but last element (reserved for PTS)

        # Extracting centroids
        blob_centroids = blob_data[:,:2] # Ignoring their area

        # Undistorting blobs centroids
        undistorted_blobs = client.camera.undistort_points(blob_centroids)          

        # Print blobs
        if verbose:
            print(f'\tDetected Blobs - {frame_idx} s')
            print('\t' + str(blob_data).replace('\n', '\n\t'))

        # Save data
        triangulator.save(ID, frame_idx, undistorted_blobs)

[SERVER] Timeout set to 5 seconds


[SERVER] Timed Out!


In [13]:
# Measured distances between perpendicularly matched marker distances
# Distances: [D_x, D_y]
wand_distances = np.array([7.5e-2, 15e-2]) # In meters

# Triangulation pair
pair = (0, 2) # Diagonal pairs seems to produce more stable results

full_vision = triangulator.full_vision()
wand_blobs = [full_vision[ID] for ID in pair]

# Update reference
server.multiple_view.update_reference(wand_blobs, wand_distances, pair)

In [14]:
# Create the Scene Viewer
scene = Viewer3D(title='Updated Camera Poses', 
                 size=10)

# Add camera frames to the scene
for ID, camera in enumerate(server.multiple_view.camera_models): 
    scene.add_frame(camera.pose, f'Camera {ID}', axis_size=0.4)

# Add new reference
scene.add_frame(np.eye(4), 'Reference', axis_size=0.4)

# Plot scene
scene.figure.show(renderer='notebook_connected')

# Saving Calibration Data

The below cell will save each camera model data as a serialized Python object. This contains all the calibration information during so far, including the intrinsic and extrinsic camera parameters, the last one regarding their current reference frame.

---

In [15]:
# Save calibration in disk
server.save_calibration()

# Standard Capture

With all calibration done, a standard capture routine can be requested for the arena's usual operation. Load and run the `data_visualizer.ttt` scene to visualize in real-time the triangulated markers.

---

In [18]:
# Capture specifications
blob_count = 4 # Number of expected markers
capture_time = 10.0 # In seconds
triangulator = Triangulator(server.multiple_view)

# Request capture (start simulation)
if not server.request_sync_capture(capture_time):
    sys.exit() # Capture request failed!

# Wait for client identification
server.register_clients()

[SERVER] Capture info sent
[SERVER] Capture confirmed!
[SERVER] Waiting for clients...
	Client 0 registered
	Client 1 registered
	Client 2 registered
	Client 3 registered
[SERVER] All clients registered!


In [19]:
verbose = False

timeout = 5 # In seconds
server.udp_socket.settimeout(timeout) # Set server timeout
print(f'[SERVER] Timeout set to {timeout} seconds\n')

visualizer_address = ('127.0.0.1', 6666)

ok_frame_idx = []
all_triangulated_markers = []

# Breaks in the timeout
while True:
    # Wait for message - Event guided!
    try:
        message_bytes, address = server.udp_socket.recvfrom(server.buffer_size)

    except TimeoutError:
        print('\n[SERVER] Timed Out!')
        break # Close capture loop due to timeout

    except ConnectionResetError:
        print('\n[SERVER] Connection Reset!')
        continue # Jump to wait for the next message
    
    # Check if message comes from any of the clients
    try:
        ID = server.client_addresses[address] # Client Identifier
    
    except:
        if verbose: print('> Address not recognized')

        continue # Jump to wait for the next message
    
    # Show sender
    if verbose: print(f'> Received message from Client {ID} ({address[0]}, {address[1]}):')

    # Decode message
    try:
        message = np.frombuffer(message_bytes, dtype=np.float32)

    except:
        if verbose: print('> Couldn\'t decode message')

        continue # Jump to wait for the next message

    # Empty message
    if not message.size:
        if verbose: print('\tEmpty message')

        continue # Jump to wait for the next message

    # Extracting the message's frame index
    frame_idx = int(message[-1])

    # Valid message is [u, v, A] per blob, PTS and frame index
    if message.size !=  3 * blob_count + 2:

        if message.size == 2:
            if verbose: print(f'\tNo blobs were detected - {frame_idx}')

        else: 
            if verbose: 
                print(f'\tWrong blob count or corrupted message')
                print(f'\tCorrupted Message: {message}')

        continue # Jump to wait for the next message

    # Extracting blob data (coordinates & area)
    blob_data = message[:-2].reshape(-1, 3) # All but last two elements

    # Extracting centroids
    blob_centroids = blob_data[:,:2] # Ignoring their area

    # Undistorting blobs centroids
    undistorted_blobs = server.clients[ID].camera.undistort_points(blob_centroids)          

    # Print blobs
    if verbose:
        print(f'\tDetected Blobs - {frame_idx}')
        print('\t' + str(blob_data).replace('\n', '\n\t'))

    triangulated_markers = triangulator.triangulate(ID, frame_idx, undistorted_blobs)

    if triangulated_markers is None:
        continue # Jump to wait for the next message

    if verbose: print("Triangulated!")

    # Send data to CoppeliaSim
    buffer = triangulated_markers.astype(np.float32).ravel().tobytes()
    server.udp_socket.sendto(buffer, visualizer_address)

    # Save data for plotting
    try:
        all_triangulated_markers.append(triangulated_markers)

    except:
        pass # Don't access array if index is out of bounds

# Join collected data
all_triangulated_markers = np.hstack(all_triangulated_markers)

[SERVER] Timeout set to 5 seconds


[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Connection Reset!

[SERVER] Con

In [20]:
# Create the Scene Viewer
scene = Viewer3D(title='Capture Profile', 
                 size=10)

# Add camera frames to the scene
for ID, camera in enumerate(server.multiple_view.camera_models): 
    scene.add_frame(camera.pose, f'Camera {ID}', axis_size=0.4)

# Add new reference
scene.add_frame(np.eye(4), 'Reference', axis_size=0.4)

# Add triangulated markers to the scene
scene.add_points(all_triangulated_markers, f'Triangulated positions')

# Plot scene
scene.figure.show(renderer='notebook_connected')